# Paper-Based IMC Workflow

## Step 1: Translate the paper into an implementation-ready workflow

This notebook starts a new, paper-faithful workflow based on the methodology in `bcd-25-0334.pdf`. We are **not** using the earlier simplified segmentation path here. Instead, we will reconstruct the analysis around the paper's preprocessing strategy:

- preprocessing with **Steinbock**
- whole-cell segmentation with **DeepCell MESMER**
- feature extraction with **Steinbock object measurements**
- downstream filtering and normalization exactly in the paper's order

For this first step, we will do only three things:

1. restate the relevant paper methodology in implementation language
2. inspect the ROI folder we want to process
3. decide whether the current ROI input is structurally compatible with a Steinbock + MESMER workflow

We will **not** move on to segmentation or preprocessing beyond inspection in this notebook step.

## What the paper does in the preprocessing and segmentation stage

From the paper's `Data analysis` section, the relevant sequence is:

1. Raw `.mcd` files are processed with the dockerized **Steinbock** toolkit.
2. Images are preprocessed with a **hot pixel filter value of 50**.
3. Whole-cell segmentation is performed with **DeepCell MESMER**.
4. The paper uses a **combined nuclear channel** made from `HistoneH3`, `191Ir`, and `193Ir`.
5. The paper uses a **combined membrane channel** made from `CD98`, `CD3`, `CD138`, and `CD45`.
6. Single-cell features are extracted with **Steinbock's object measurement module**.
7. Cells smaller than **4 pixels** are removed.
8. Each marker is censored to the **99th percentile**.
9. Marker intensities are **arcsinh-transformed with cofactor 1**.
10. For downstream analysis, each cell is **CLR-normalized** using total protein abundance per cell.

This gives us an important practical constraint:

A paper-grade implementation is not just "run a segmentation model". It requires us to prepare the input exactly as Steinbock and MESMER expect, including the correct nuclear and membrane composites.

## What we need to verify before Step 2

Because you are providing an already extracted ROI folder rather than the original `.mcd`, we need to answer a key question before we proceed:

**Can this ROI folder be treated as a valid Steinbock-ready image bundle?**

To answer that, we need to inspect:

- which TIFF files are present
- which markers are available
- whether the paper's required MESMER input channels are available in your ROI
- whether all images have matching dimensions and pixel types

This step is important because if the paper's exact segmentation channels are missing, we should pause and decide whether to:

- reproduce the paper as closely as possible with substitute channels, or
- stop and require upstream export changes

That decision should be explicit, because it changes how faithfully we can claim to follow the paper.

In [1]:
from pathlib import Path
import csv
from collections import Counter

try:
    from PIL import Image
except ImportError as exc:
    raise RuntimeError('Pillow is required for this notebook step.') from exc

WORKFLOW_ROOT = Path('/Users/rashid/1_IMC_Analysis/11_Vincenzo/paper_based_workflow')
ROI_DIR = Path('/Users/rashid/1_IMC_Analysis/11_Vincenzo/ROI001_D13')
OUTPUT_DIR = WORKFLOW_ROOT / 'step1_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

channel_files = sorted(list(ROI_DIR.glob('*.tif')) + list(ROI_DIR.glob('*.tiff')))
print(f'ROI folder: {ROI_DIR}')
print(f'Number of channel files: {len(channel_files)}')


ROI folder: /Users/rashid/1_IMC_Analysis/11_Vincenzo/ROI001_D13
Number of channel files: 43


In [2]:
def parse_marker_name(path: Path) -> dict:
    stem = path.name.replace('.ome.tiff', '').replace('.ome.tif', '')
    if '_' in stem:
        metal_tag, marker = stem.split('_', 1)
    else:
        metal_tag, marker = 'UNKNOWN', stem
    return {
        'filename': path.name,
        'metal_tag': metal_tag,
        'marker': marker,
    }

records = []
shape_counter = Counter()
dtype_counter = Counter()

for path in channel_files:
    record = parse_marker_name(path)
    with Image.open(path) as img:
        record['width'], record['height'] = img.size
        record['mode'] = img.mode
        record['dtype_hint'] = getattr(getattr(img, 'getbands', lambda: ())(), '__class__', tuple).__name__
    shape_counter[(record['width'], record['height'])] += 1
    dtype_counter[record['mode']] += 1
    records.append(record)

inventory_csv = OUTPUT_DIR / 'step1_channel_inventory.csv'
with inventory_csv.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(records[0].keys()) if records else [])
    writer.writeheader()
    writer.writerows(records)

print(f'Inventory saved to: {inventory_csv}')
print('Image sizes found:', dict(shape_counter))
print('Image modes found:', dict(dtype_counter))


Inventory saved to: /Users/rashid/1_IMC_Analysis/11_Vincenzo/paper_based_workflow/step1_outputs/step1_channel_inventory.csv
Image sizes found: {(1000, 1000): 43}
Image modes found: {'I;16': 43}


In [3]:
available_markers = sorted(record['marker'] for record in records)
print('Available markers:')
for marker in available_markers:
    print('-', marker)

paper_nuclear = ['HistoneH3', '191Ir', '193Ir']
paper_membrane = ['CD98', 'CD3', 'CD138', 'CD45']

available_set = set(available_markers)
present_nuclear = [m for m in paper_nuclear if m in available_set]
missing_nuclear = [m for m in paper_nuclear if m not in available_set]
present_membrane = [m for m in paper_membrane if m in available_set]
missing_membrane = [m for m in paper_membrane if m not in available_set]

print('\nPaper nuclear channel requirements')
print('Present:', present_nuclear)
print('Missing:', missing_nuclear)

print('\nPaper membrane channel requirements')
print('Present:', present_membrane)
print('Missing:', missing_membrane)


Available markers:
- 127I
- 131Xe
- 134Xe
- 138Ba
- 208Pb
- 80ArAr
- BCL2
- Brachiury
- CD10
- CD11c
- CD138
- CD163
- CD20
- CD21
- CD3
- CD31
- CD4
- CD44
- CD45RO
- CD47
- CD56
- CD66b
- CD68
- CD72a
- CD8
- CD80
- CD86
- CTLA4
- DNA1
- DNA2
- ERG
- FoxP3
- GATA3
- GranzymeB
- Ki67
- LAG3
- MHC_II
- PD1
- PDL1
- S100
- TIM3
- Tbet
- aSMA

Paper nuclear channel requirements
Present: []
Missing: ['HistoneH3', '191Ir', '193Ir']

Paper membrane channel requirements
Present: ['CD3', 'CD138']
Missing: ['CD98', 'CD45']


## How to interpret Step 1

After you run the code above, focus on these questions:

### 1. Are all TIFFs aligned?
If all files have the same width and height, that is a good sign. Steinbock-style downstream measurement assumes that all channels refer to the same spatial grid.

### 2. Do we have the paper's exact MESMER input channels?
This is the most important checkpoint. The paper uses:

- nuclear: `HistoneH3 + 191Ir + 193Ir`
- membrane: `CD98 + CD3 + CD138 + CD45`

If one or more of these are missing in your ROI export, then we cannot truthfully say we reproduced the segmentation stage exactly. We can still build a high-quality **paper-inspired** segmentation workflow, but that becomes a documented adaptation rather than a strict reproduction.

### 3. Why we are stopping here
Before moving into actual preprocessing, we need your agreement on how strict you want to be.

There are only two scientifically honest paths:

- **Strict path**: require the paper's exact channels and input structure
- **Adapted path**: use the best available substitutes and document the deviation clearly

We should choose that explicitly before Step 2.

## Stop point

This notebook intentionally stops after Step 1.

Once you run it and inspect the marker availability, we can decide together whether Step 2 should be:

- preparing a Steinbock-compatible image layout with the exact paper channels, or
- designing a documented substitute-channel strategy for MESMER input

We should not proceed until that decision is made.